**1. Install Libraries**

In [1]:
# Install required libraries
!pip install datasets pandas sqlglot transformers torch tqdm scikit-learn

**2. Import Libraries**

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import sqlglot
from tqdm import tqdm

from sklearn.model_selection import train_test_split

from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

**3. Load PARROT Dataset**

In [3]:
# Load PARROT dataset from HuggingFace
dataset = load_dataset("weizhoudb/PARROT")

# Convert to pandas
df = dataset["test"].to_pandas()

print("Total dataset size:", len(df))
df.head()

README.md: 0.00B [00:00, ?B/s]

parrot_diverse.json:   0%|          | 0.00/26.7M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/28003 [00:00<?, ? examples/s]

Total dataset size: 28003


,norm,sqlite,oracle,id,tsql,athena,bigquery,clickhouse,doris,drill,...,mysql,postgres,presto,redshift,risingwave,snowflake,spark,starrocks,teradata,trino
0,SELECT DISTINCT identifier FROM identifier AS ...,SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AI...,SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AI...,ATIS,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,SELECT DISTINCT identifier FROM identifier AS ...,SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM FL...,None,ATIS,SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM FL...,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,SELECT DISTINCT identifier FROM identifier AS ...,SELECT DISTINCT FAREalias0.FARE_ID FROM AIRPOR...,None,ATIS,None,SELECT DISTINCT FAREalias0.FARE_ID FROM AIRPOR...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,SELECT DISTINCT identifier FROM identifier AS ...,SELECT DISTINCT AIRLINEalias0.AIRLINE_CODE FRO...,None,ATIS,None,None,SELECT DISTINCT AIRLINEalias0.AIRLINE_CODE FRO...,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,SELECT DISTINCT identifier FROM identifier AS ...,SELECT DISTINCT FAREalias0.FARE_ID FROM AIRPOR...,None,ATIS,None,None,None,SELECT DISTINCT FAREalias0.FARE_ID FROM AIRPOR...,None,None,...,None,None,None,None,None,None,None,None,None,None


**4. Extract SQLite → Hive Pairs**

In [4]:
# Select only SQLite and Hive columns and drop missing values
sqlite_hive_df = df[["sqlite", "hive"]].dropna()

print("SQLite → Hive pairs:", len(sqlite_hive_df))
sqlite_hive_df.head()

SQLite → Hive pairs: 714


,sqlite,hive
10,SELECT DISTINCT FAREalias0.FARE_ID FROM FARE A...,SELECT DISTINCT FAREalias0.FARE_ID FROM FARE A...
31,SELECT COUNT( DISTINCT FLIGHTalias0.FLIGHT_ID ...,SELECT COUNT(DISTINCT FLIGHTalias0.FLIGHT_ID) ...
60,SELECT STATEalias0.STATE_NAME FROM STATE AS ST...,SELECT STATEalias0.STATE_NAME FROM STATE AS ST...
82,SELECT DISTINCT RIVERalias0.RIVER_NAME FROM RI...,SELECT DISTINCT RIVERalias0.RIVER_NAME FROM RI...
134,SELECT COUNT( DISTINCT RIVERalias0.RIVER_NAME ...,SELECT COUNT(DISTINCT RIVERalias0.RIVER_NAME) ...


**5. Dataset Cleaning (Milestone 2 reused)**

5.1 Remove Duplicates

In [5]:
duplicate_count = sqlite_hive_df.duplicated(subset=["sqlite", "hive"]).sum()
print("Duplicates:", duplicate_count)

sqlite_hive_df = sqlite_hive_df.drop_duplicates(subset=["sqlite", "hive"])

Duplicates: 46


5.2 Remove Malformed SQL using sqlglot

In [6]:
invalid_indices = []

for i, row in tqdm(sqlite_hive_df.iterrows(), total=len(sqlite_hive_df)):
    try:
        sqlglot.parse_one(row["sqlite"], read="sqlite")
        sqlglot.parse_one(row["hive"], read="hive")
    except:
        invalid_indices.append(i)

print("Invalid queries:", len(invalid_indices))

sqlite_hive_df = sqlite_hive_df.drop(index=invalid_indices)

100%|██████████| 668/668 [00:08<00:00, 83.09it/s] 

Invalid queries: 1


5.3 Normalize Text (Encoding + Formatting)

In [7]:
def normalize_sql(query):
    return str(query).strip().lower()

sqlite_hive_df["sqlite"] = sqlite_hive_df["sqlite"].apply(normalize_sql)
sqlite_hive_df["hive"] = sqlite_hive_df["hive"].apply(normalize_sql)

**6. Train / Validation / Test Split**

In [8]:
train_df, temp_df = train_test_split(sqlite_hive_df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 466
Validation: 100
Test: 101


**7. Tokenization (Model Input Preparation)**

In [9]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Max token length
MAX_LEN = 128

def tokenize_data(input_text, target_text):
    input_enc = tokenizer(
        input_text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    target_enc = tokenizer(
        target_text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    return input_enc, target_enc

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**8. Load Model (Architecture)**

In [10]:
# Load pretrained T5 model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on: cpu


**9. Small Pipeline (Milestone Requirement)**

👉 We use only 10 samples to demonstrate full pipeline

In [11]:
sample_data = test_df.head(10)

sample_data

,sqlite,hive
13741,"select t1.app, count(t1.app) countnumber from ...","select t1.app, count(t1.app) as countnumber fr..."
2875,"select t1.name , t3.visit_date from tourist_at...","select t1.name, t3.visit_date from tourist_att..."
10793,"select distinct t1.diagnosis, t1.id , strftime...","select distinct t1.diagnosis, t1.id, date_form..."
19545,"select n_2009 , n_2010 + 5 , abs ( n_2010 + 5 ...","select n_2009, n_2010 + 5, abs(n_2010 + 5 - n_..."
15083,select cast(sum(t2.`order quantity`) as real) ...,select cast(sum(t2.`order quantity`) as float)...
6829,select t2.description from works as t1 inner j...,select t2.description from works as t1 inner j...
6861,select sum(t1.amount) from payment as t1 inner...,select sum(t1.amount) from payment as t1 inner...
15560,select sum(case when cast(replace(trim(inflati...,select sum(case when cast(replace(trim(inflati...
7006,"select t3.first_name, t3.last_name from ( sele...","select t3.first_name, t3.last_name from (selec..."
8320,select cast(sum(case when t2.job_desc in ('edi...,select cast(sum(case when t2.job_desc in ('edi...


**10. Run End-to-End Pipeline**

In [12]:
model.eval()

results = []

for i, row in sample_data.iterrows():

    input_text = "translate sqlite to hive: " + row["sqlite"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    ).to(device)

    # Generate output
    output_ids = model.generate(
        inputs["input_ids"],
        max_length=MAX_LEN
    )

    predicted = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    results.append({
        "SQLite": row["sqlite"],
        "Actual Hive": row["hive"],
        "Predicted Hive": predicted
    })

**11. Display Model Outputs**

In [15]:
def clean_prediction(text):
    text = text.lower().strip()

    # Remove task prefix if present
    text = text.replace("translate sqlite to hive:", "")
    text = text.replace("sqlite to hive:", "")

    # Normalize spaces
    text = " ".join(text.split())

    return text


results = []

for i, row in sample_data.iterrows():

    input_text = "translate sqlite to hive: " + row["sqlite"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    ).to(device)

    output_ids = model.generate(
        inputs["input_ids"],
        max_length=MAX_LEN
    )

    predicted = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # ✅ CLEAN HERE
    predicted_clean = clean_prediction(predicted)
    actual_clean = clean_prediction(row["hive"])

    results.append({
        "SQLite": row["sqlite"],
        "Actual Hive": actual_clean,
        "Predicted Hive": predicted_clean
    })

**12. Evaluate (Basic Metric)**
Exact Match Accuracy

In [18]:
from difflib import SequenceMatcher

def similarity_score(a, b):
    return SequenceMatcher(None, a, b).ratio()

results_df["similarity"] = results_df.apply(
    lambda x: similarity_score(x["Predicted Hive"], x["Actual Hive"]),
    axis=1
)

print("Average Similarity Score:", results_df["similarity"].mean())

Average Similarity Score: 0.7814680414903553


**The similarity score of around 0.78 shows that the generated queries are structurally and semantically close to the expected Hive queries. This indicates that the pipeline is working correctly, but the model requires fine-tuning for production-level accuracy.**

**13. Save Outputs**

In [19]:
results_df.to_csv("model_outputs.csv", index=False)
print("Saved outputs")

Saved outputs
